# Demucs

In [1]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch
import torchaudio
from clear_memory import clear_memory

warnings.filterwarnings('ignore')

In [2]:
input_dir_Pitt = Path('../ad_detection/data/raw/Pitt')
output_dir_Pitt = Path('../ad_detection/data/denoised/Pitt-Demucs')

control_files_Pitt = list((input_dir_Pitt / 'Control').glob('*.wav')) + list((input_dir_Pitt / 'Control').glob('*.mp3'))
dementia_files_Pitt = list((input_dir_Pitt / 'Dementia').glob('*.wav')) + list((input_dir_Pitt / 'Dementia').glob('*.mp3'))

input_dir_Lu = Path('../ad_detection/data/raw/Lu')
output_dir_Lu = Path('../ad_detection/data/denoised/Lu-Demucs')

control_files_Lu = list((input_dir_Lu / 'Control').glob('*.wav')) + list((input_dir_Lu / 'Control').glob('*.mp3'))
dementia_files_Lu = list((input_dir_Lu / 'Dementia').glob('*.wav')) + list((input_dir_Lu / 'Dementia').glob('*.mp3'))

## Load Demucs Model

In [3]:
from demucs.pretrained import get_model
from demucs.apply import apply_model

model_name = 'htdemucs_ft'

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

torch.device(device)

model = get_model(model_name)
model.to(device)
model.eval()

BagOfModels(
  (models): ModuleList(
    (0-3): 4 x HTDemucs(
      (encoder): ModuleList(
        (0): HEncLayer(
          (conv): Conv2d(4, 48, kernel_size=(8, 1), stride=(4, 1), padding=(2, 0))
          (norm1): Identity()
          (rewrite): Conv2d(48, 96, kernel_size=(1, 1), stride=(1, 1))
          (norm2): Identity()
          (dconv): DConv(
            (layers): ModuleList(
              (0): Sequential(
                (0): Conv1d(48, 6, kernel_size=(3,), stride=(1,), padding=(1,))
                (1): GroupNorm(1, 6, eps=1e-05, affine=True)
                (2): GELU(approximate='none')
                (3): Conv1d(6, 96, kernel_size=(1,), stride=(1,))
                (4): GroupNorm(1, 96, eps=1e-05, affine=True)
                (5): GLU(dim=1)
                (6): LayerScale()
              )
              (1): Sequential(
                (0): Conv1d(48, 6, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
                (1): GroupNorm(1, 6, eps=1e-05, affine=Tr

## Denoise Function

In [4]:
def denoise_audio(audio_path, model, device):
    """
    使用 Demucs 进行语音分离和增强
    从音频中提取vocals（人声）部分，去除背景噪音、音乐等干扰
    注意：Demucs将所有人声（讲话+唱歌）都归类为vocals
    
    Args:
        audio_path: 输入音频文件路径
        model: Demucs 模型实例
        device: 计算设备 (cuda/mps/cpu)
    
    Returns:
        vocals_audio: 提取的人声音频 numpy array（单声道）
        sr: 采样率
    """
    # 加载音频
    audio, sr = sf.read(str(audio_path))
    
    # Demucs 需要 2 通道（立体声）输入
    # 处理不同声道数的音频
    if len(audio.shape) == 1:
        # 单声道：复制为双声道 [time] -> [time, 2]
        audio = np.stack([audio, audio], axis=1)
    elif len(audio.shape) == 2:
        if audio.shape[1] == 1:
            # [time, 1] -> [time, 2]
            audio = np.concatenate([audio, audio], axis=1)
        elif audio.shape[1] > 2:
            # 多于2个通道，只取前2个
            audio = audio[:, :2]
        # 如果已经是2通道，保持不变
    
    # 重采样到模型要求的采样率
    target_sr = model.samplerate
    if sr != target_sr:
        # 对每个通道分别重采样
        num_samples = int(audio.shape[0] * target_sr / sr)
        audio_resampled = np.zeros((num_samples, 2), dtype=audio.dtype)
        for ch in range(2):
            audio_resampled[:, ch] = signal.resample(audio[:, ch], num_samples)
        audio = audio_resampled
        sr = target_sr
    
    # 转换为 torch tensor
    # Demucs 期望输入形状为 [batch, channels, time]
    # audio 当前形状: [time, 2]
    audio_tensor = torch.from_numpy(audio.T).float()  # [2, time]
    audio_tensor = audio_tensor.unsqueeze(0)  # [1, 2, time]
    audio_tensor = audio_tensor.to(device)
    
    # 应用 Demucs 源分离
    with torch.no_grad():
        # apply_model 返回分离后的音频源
        # 输出形状: [batch, sources, channels, time]
        # sources 顺序通常为: ['drums', 'bass', 'other', 'vocals']
        sources = apply_model(
            model, 
            audio_tensor, 
            device=device,
            split=True,  # 分段处理，节省显存
            overlap=0.25  # 重叠25%以避免边界效应
        )
    
    # 提取 vocals 源（包含所有人声：讲话+唱歌）
    # 找到 vocals 在源列表中的索引
    try:
        vocals_idx = model.sources.index('vocals')
    except (AttributeError, ValueError):
        # 如果找不到，假设是最后一个源
        vocals_idx = -1
    
    # 转换回 numpy，并转为单声道
    # sources 形状: [batch, sources, channels, time]
    # 取双通道的平均值作为单声道输出
    vocals_stereo = sources[0, vocals_idx, :, :].cpu().numpy()  # [2, time]
    vocals_audio = np.mean(vocals_stereo, axis=0)  # [time] 单声道
    
    return vocals_audio, sr

In [5]:
def batch_denoise(files, output_subdir, model, device, group_name):
    """
    批量音频源分离处理（每个文件前后都清理显存）
    提取vocals（人声）部分，去除背景噪音
    
    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        model: Demucs 模型实例
        device: 计算设备
        group_name: 组名（用于显示进度）
    """
    # 创建输出目录
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    skip_count = 0
    fail_count = 0
    
    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        # 统一输出为 .wav 格式
        output_file = output_subdir / (audio_file.stem + '.wav')
        
        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue
        
        try:
            # ⚡ 处理前清理显存
            clear_memory()
            
            # 源分离，提取vocals（人声）
            vocals_audio, sr = denoise_audio(audio_file, model, device)
            
            # 保存（16位整数格式）
            sf.write(str(output_file), vocals_audio, sr, subtype='PCM_16')
            success_count += 1
            
            # ⚡ 处理后立即清理显存
            del vocals_audio  # 删除大数组
            clear_memory()
            
        except Exception as e:
            fail_count += 1
            print(f"\n✗ 处理失败: {audio_file.name}: {e}")
            # ⚡ 失败后也要清理显存
            clear_memory()
    
    # 打印统计信息
    print(f"\n{group_name} 处理完成:")
    print(f"  成功: {success_count}")
    print(f"  跳过: {skip_count}")
    print(f"  失败: {fail_count}")

## Pitt Denoise

In [6]:
batch_denoise(
    dementia_files_Pitt,
    output_dir_Pitt / 'Dementia',
    model,
    device,
    group_name='Dementia'
)

# Clear memory between group s
clear_memory()

batch_denoise(
    control_files_Pitt,
    output_dir_Pitt / 'Control',
    model,
    device,
    group_name='Control'
)

Processing Dementia: 100%|██████████| 309/309 [00:00<00:00, 56369.17it/s]



Dementia 处理完成:
  成功: 0
  跳过: 309
  失败: 0


Processing Control: 100%|██████████| 242/242 [00:00<00:00, 101077.63it/s]


Control 处理完成:
  成功: 0
  跳过: 242
  失败: 0


## Lu Denoise

In [7]:
batch_denoise(
    dementia_files_Lu,
    output_dir_Lu / 'Dementia',
    model,
    device,
    group_name='Dementia'
)

# Clear memory between group s
clear_memory()

batch_denoise(
    control_files_Lu,
    output_dir_Lu / 'Control',
    model,
    device,
    group_name='Control'
)

Processing Dementia: 100%|██████████| 38/38 [00:00<00:00, 66216.68it/s]



Dementia 处理完成:
  成功: 0
  跳过: 38
  失败: 0


Processing Control: 100%|██████████| 36/36 [00:00<00:00, 44397.22it/s]


Control 处理完成:
  成功: 0
  跳过: 36
  失败: 0
